# 使用 PROC CHART 剖析区域电网负荷与停电情况

## 执行摘要

一位电网运营分析师使用 PROC CHART 剖析一份合成样本数据，该数据涵盖四个供电区域和四种发电来源的馈线回路电表读数。本笔记本依次演示垂直条形图、水平条形图、块状图和饼图，用以汇总发电构成、按区域比较平均负荷与总负荷、揭示傍晚需求高峰的小时分布，并按来源对停电分钟数进行排序 —— 这正是能源与公用事业团队在投入制作图形化报告之前所进行的那种快速的文本输出式探索。DATA 步请求生成 1,200 行；本笔记本以未授权模式渲染，工作数据集被限制在前 100 条读数，因此下面每张图表都汇总的是这份 100 行的样本。

## 数据来源

| 数据集 | 行数 | 说明 |
|---------|------|-------------|
| `grid_ops` | 100（合成样本） | 区域电网上每条馈线回路电表读数占一行，通过 `call streaminit(20260531)` 和 `rand()` 内联生成。DATA 步循环请求 1,200 条读数，但未授权模式将保存的数据集限制在 100 条观测，因此下面的图表描述的是这 100 行。 |

# 使用 PROC CHART 剖析区域电网负荷与停电情况

PROC CHART 直接在清单中生成基于字符的条形图、块状图和饼图 —— 无需 ODS Graphics 设备。这使它成为一个快速的初步探索工具，适合电网运营团队在构建精致的 GCHART 或 SGPLOT 可视化之前，先*看清*其负荷和可靠性数据的形态。

在本笔记本中，我们将：

1. 为区域电网运营商生成一个月的合成馈线回路电表读数。
2. 绘制**发电构成**（按来源统计读数占比）。
3. 比较各供电区域的**平均与总交付负荷**。
4. 揭示按一天中的小时分布的**傍晚需求高峰**。
5. 使用**块状图**将区域与发电来源交叉分析。
6. 按来源对**停电分钟数**排序，找出最不可靠的馈线。

下面的每一条语句和选项都是标准的 SAS 9.4 PROC CHART 语法。

## 步骤 1 —— 生成合成运营数据

下面的 DATA 步在一个 1,200 次迭代的循环中编造电表读数。每条读数都被分配一个供电区域、一个发电来源（Gas 占主导，其余由 Wind、Solar 和 Hydro 构成）以及一天中的某个小时。在 17:00–21:00 的傍晚时段负荷更高（我们将这些读数标记为高峰），并从泊松分布中抽取停电分钟数。`call streaminit` 固定随机种子，使数据可复现。

日志中的 NOTE 显示了实际结果：本次运行处于未授权模式，因此保存的 `grid_ops` 数据集被限制在这些读数中的前 100 条（100 行，7 列）。随后的每张图表都汇总这份 100 行的样本，且每条 PROC CHART 日志都确认读取了 100 行。

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
数据 grid_ops;
    调用 streaminit(20260531);
    长度 region $12 source $9 peak_flag $3;
    数组 regions[4] $12 _temporary_
        ('North','South','East','West');
    循环 meter_id = 1 到 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        如果 u < 0.42 那么 source = 'Gas';
        否则 如果 u < 0.70 那么 source = 'Wind';
        否则 如果 u < 0.88 那么 source = 'Solar';
        否则 source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        如果 hour >= 17 并且 hour <= 21 那么 循环;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        结束;
        否则 循环;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        结束;
        如果 load_mwh < 0 那么 load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        输出;
    结束;
    删除 r u BASE;
运行;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## 步骤 2 —— 发电构成

每种发电来源占了多少比例的读数？带 `TYPE=PERCENT` 的垂直条形图可直接回答这个问题：条形高度是落入每个来源类别的全部观测的百分比。由于 `source` 是字符变量，PROC CHART 会自动将其取值视为离散类别。

In [2]:
过程 chart 数据=grid_ops;
    VBAR source / type=PERCENT;
运行;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 步骤 3 —— 各区域的平均交付负荷

现在我们从计数转向汇总某个测量值。在 `SUMVAR=` 中指定 `load_mwh` 并配合 `TYPE=MEAN`，使条形长度表示各区域的平均负荷，同时我们显式请求统计列：`MEAN` 在每个条形旁打印平均值，`CFREQ` 添加一个累积频率列。North 的平均交付负荷最高（均值约 28.0 MWh），与数据中内置的区域偏移一致，而 South 最低（约 19.8 MWh）。

我们还传入了 `ASCENDING`，意图将条形按从低到高的均值排序。请注意输出中条形**并未**重新排序 —— 它们仍按类别顺序显示（West、South、East、North），其均值 24.2、19.8、21.7、28.0 并非从短到长排列。按图表统计量重排条形在本 PROC CHART 实现中尚未接通，因此 `ASCENDING`/`DESCENDING` 虽被接受，但目前不产生任何效果；请改从打印出的 `Mean` 列读取排名。

In [3]:
过程 chart 数据=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
运行;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 步骤 4 —— 各区域的总负荷

这里 `TYPE=SUM` 使每个条形表示该区域的*总*交付负荷而非平均值，因此较高的条形标记出在采样读数中交付总能量最多的区域。North 在总交付负荷上再次领先。

该语句还请求了 `SUBGROUP=peak_flag`，要求 PROC CHART 按读数是否落在傍晚高峰时段来拆分每个条形。在 SAS 中，这会用每个子组级别对应的不同字形来分段每个条形并打印图例。本实现尚未渲染子组分段（这是一项已记录的能力缺口），因此条形以实心的 `*` 连续填充呈现，不显示逐段拆分 —— 条形*总计*是正确的，但此处不显示高峰与非高峰的拆分。若要查看有多少负荷落在高峰时段，请使用步骤 5 中按小时分布的视图。

In [4]:
过程 chart 数据=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
运行;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 步骤 5 —— 一天中的负荷形态

`hour` 是连续变量，因此我们通过显式的 `MIDPOINTS=` 列表在 4 小时为中心处（2、6、10、14、18、22）自行固定分箱。每个条形显示接近该小时的读数的平均交付负荷。以 18 为中心的条形应当格外突出 —— 那正是我们在数据中内置的傍晚需求高峰。

In [5]:
过程 chart 数据=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
运行;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## 步骤 6 —— 区域与发电来源交叉（块状图）

BLOCK 图将一个小的双向表渲染为块状网格。通过 `GROUP=source` 和 `SUMVAR=load_mwh / TYPE=MEAN`，该图将每个区域与为其供电的发电来源交叉分析，块高与平均负荷成正比 —— 这是一种紧凑的方式，可用于发现哪些区域/来源组合承载了最重的平均负荷。

In [6]:
过程 chart 数据=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
运行;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 步骤 7 —— 以饼图呈现发电构成

与步骤 2 相同的来源占比信息，以饼图绘制。带 `TYPE=PERCENT` 的 PIE 按每个扇区占总读数的百分比确定其大小，并在图旁打印各扇区百分比的图例。

In [7]:
过程 chart 数据=grid_ops;
    PIE source / type=PERCENT;
运行;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 步骤 8 —— 按来源统计停电分钟数

最后我们对可靠性进行排名。`SUMVAR=outage_min` 配合 `TYPE=SUM` 汇总每个来源的停电分钟数。我们传入 `DESCENDING`，试图将表现最差的来源浮到顶部，但与步骤 3 一样，条形并未重新排序 —— 它们按类别顺序打印（Hydro、Wind、Gas、Solar），且条形重排尚未实现。请从打印出的 `Sum` 列读取排名：在本样本中，Gas 的停电分钟数总计最多（122），远超 Wind（64）、Solar（43）和 Hydro（38）。这与发电构成一致 —— Gas 服务的读数最多，因此累积的停电分钟数总体上也最多。

In [8]:
过程 chart 数据=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDING;
运行;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 结果解读

将这些图表结合起来阅读，可为运营团队提供一幅快速的态势画面（基于本次运行捕获的 100 行样本）：

- **发电构成（步骤 2 和 7）。** Gas 占读数的最大份额（约 45%），Wind 居第二（约 28%），Hydro 和 Solar 靠后（约 14% 和 13%）—— 垂直条形图和饼图以两种方式讲述同一个故事，是一次有用的合理性检查。
- **各区域负荷（步骤 3 和 4）。** 平均负荷的 HBAR 显示 North 的平均交付负荷最高（均值约 28 MWh），South 最低（约 20 MWh），与数据中内置的区域偏移一致。SUM 图表进一步确认 North 交付的总能量也最多。
- **每日负荷形态（步骤 5）。** 以小时 18 为中心的中点条形明显最高，确认了我们在数据中内置的 17:00–21:00 需求高峰 —— 这正是公用事业公司会重点关注需求响应和容量规划的时段。
- **可靠性（步骤 8）。** 按来源汇总停电分钟数，使 Gas 显现为本样本中停机时间的最大贡献者（122 分钟），成为维护审查的自然下一目标 —— 尽管这在很大程度上反映了 Gas 服务的读数最多。

此处使用的两个显示选项 —— `ASCENDING`/`DESCENDING` 条形重排（步骤 3 和 8）以及 `SUBGROUP=` 条形分段（步骤 4）—— 被解析器接受，但本 PROC CHART 实现尚未渲染，因此排名和拆分应从打印出的统计列而非条形顺序或阴影中读取。

PROC CHART 仅输出字符，因此若要制作可供董事会汇报的可视化图形，团队会将这些相同的视图迁移到 PROC GCHART 或 PROC SGPLOT。但作为对新提取数据的零配置初步扫描，这些图表能在数秒之内回答运营性问题 —— 构成、量级和时机。